In [2]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

REPO = "smart-waste-multi-agent"

%cd /content
!git clone https://{token}@github.com/Nafila1234/{REPO}.git
%cd /content/{REPO}
!git config user.name "Nafila1234"
!git config user.email "nafiladziri17@gmail.com"

/content
Cloning into 'smart-waste-multi-agent'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 136 (delta 53), reused 49 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.00 MiB | 14.03 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/smart-waste-multi-agent


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive/DM_Project

Mounted at /content/drive
vision_agent_resnet18_dedup.pth      vision_agent_resnet18.pth
vision_agent_resnet18_finetuned.pth


In [4]:
%%writefile /content/smart-waste-multi-agent/visual_agent/vision_agent.py
"""Vision agent: classifies a waste photo into one of 6 classes."""
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image

CLASSES = ["battery", "glass", "metal", "organic", "paper", "plastic"]
_device = "cuda" if torch.cuda.is_available() else "cpu"
_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def load_model(weights_path):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    m.load_state_dict(torch.load(weights_path, map_location=_device))
    return m.to(_device).eval()

def predict_waste(image, model):
    """image: PIL image or file path. Returns (class_name, confidence)."""
    if isinstance(image, str):
        image = Image.open(image)
    x = _tf(image.convert("RGB")).unsqueeze(0).to(_device)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    i = int(probs.argmax())
    return CLASSES[i], float(probs[i])

Writing /content/smart-waste-multi-agent/visual_agent/vision_agent.py


In [10]:
import sys, os
sys.path.append("/content/smart-waste-multi-agent/visual_agent")
from vision_agent import load_model, predict_waste

agent = load_model("/content/drive/MyDrive/DM_Project/vision_agent_resnet18_dedup.pth")
print(predict_waste("/content/images.jpg", agent))

('glass', 0.9764686822891235)


In [ ]:
%cd /content/smart-waste-multi-agent
!git pull
!git add .
!git commit -m "Add vision agent function"
!git push